# jev-my-bro — Laya RLCD training on Google Colab

This notebook trains the active v0.2 Laya-based model. The repository already contains the static dataset; no dataset generation happens here.

Before running: Runtime → Change runtime type → GPU. Put the repository folder in Google Drive at `MyDrive/jev-my-bro` or change `SOURCE` below.

In [2]:
import shutil
import subprocess
import torch

if not torch.cuda.is_available():
    print('CUDA is unavailable in this Colab runtime.')
    print('Select Runtime -> Change runtime type -> T4 GPU, then reconnect and rerun this cell.')
    if shutil.which('nvidia-smi') is None:
        print('Diagnostic: nvidia-smi is not installed, so this is a CPU runtime.')
    raise RuntimeError('A Colab GPU runtime is required for RLCD fine-tuning.')

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], check=False, capture_output=True, text=True).stdout)
print('GPU:', torch.cuda.get_device_name(0))

Mon Sep 21 03:37:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path
import os, shutil

CLI_SOURCE = Path('/content/jev-my-bro')
if (CLI_SOURCE / 'requirements.txt').exists():
    SOURCE = CLI_SOURCE
    REPO = Path('/content/jev-my-bro-run')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SOURCE = Path('/content/drive/MyDrive/jev-my-bro')
    REPO = Path('/content/jev-my-bro')
assert SOURCE.exists(), f'Upload the repo folder to {SOURCE} or edit SOURCE'
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SOURCE, REPO, ignore=shutil.ignore_patterns('artifacts', '__pycache__', '.git'))
os.chdir(REPO)
print('working directory:', Path.cwd())

working directory: /content/jev-my-bro-run


In [ ]:
!pip install -q -r requirements.txt
import laya
print('laya:', laya.__version__)

laya: 0.3.4


In [ ]:
!python scripts/validate_dataset.py

train       cases= 576 decisions=2304 languages={'en': 384, 'th': 192} actions={'ask_user': 192, 'execute': 192, 'reject': 192}
validation  cases= 144 decisions= 576 languages={'th': 48, 'en': 96} actions={'ask_user': 48, 'reject': 48, 'execute': 48}
calibration cases= 144 decisions= 576 languages={'en': 96, 'th': 48} actions={'execute': 48, 'reject': 48, 'ask_user': 48}
test        cases= 144 decisions= 576 languages={'en': 96, 'th': 48} actions={'reject': 48, 'execute': 48, 'ask_user': 48}
dataset validation: OK


## Train

The default base is `convaiinnovations/laya-multilingual` because this dataset contains English and Thai. The trainer is adapted for one Colab GPU using gradient accumulation.

In [ ]:
!python -m jevbro.train \
  --train data/train.jsonl \
  --validation data/validation.jsonl \
  --base-model convaiinnovations/laya-multilingual \
  --output artifacts/laya-model \
  --epochs 4 \
  --micro-batch 4 \
  --grad-accum 8 \
  --group-size 4

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.



Reconstructing (incomplete total...):   0% 0.00/1.58k [00:00<?, ?B/s]         

Fetching 7 files:  14% 1/7 [00:00<00:00,  8.96it/s]
Reconstructing (incomplete total...):  19% 1.58k/8.54k [00:00<00:00, 14.4kB/s]
Reconstructing (incomplete total...):  94% 8.54k/9.04k [00:00<00:00, 14.4kB/s]
Reconstructing (incomplete total...):   0% 9.04k/644M [00:00<12:27:03, 14.4kB/s]
Reconstructing (incomplete total...):   0% 9.04k/678M [00:00<13:06:55, 14.4kB/s]
Reconstructing (incomplete total...):   0% 9.04k/678M [00:00<13:06:55, 14.4kB/s]

Fetching 7 files:  57% 4/7 [00:00<00:00, 20.25it/s]
Reconstructing (incomplete total...):   0% 9.51k/678M [00:00<3:48:26, 49.5kB/s]

Reconstructing (incomplete total...):  45% 303M/678M [00:02<00:02, 178MB/s,  145MB/s  ]


Reconstructing (incomplete total...): 100% 678M/678M [00:05<00:00, 116MB/s,  178MB/s  ]

Fetching 7 files: 100% 7/7 [00:05<00:00,  1.21it/s]
Download complete: 100% 653M/653M [00:05<00:00, 540MB/s,  540MB/s  ]
Reconstruction complete: 100% 678M/678M [00:05<00:00, 116MB/s,  116MB/s  ]             

Download complete: 100% 653M/653M [00:10<00:00, 65.1MB/s,  540MB/s  ]
Reconstruction complete: 100% 678M/678M [00:10<00:00, 67.6MB/s,  116MB/s  ]


...

{
  "epoch": 1,
  "epochs": 4,
  "loss": 0.4481149542067821,
  "reward": -0.08451157621392566,
  "sigma": 0.4,
  "validation": {
    "accuracy": 0.8975694444444444,
    "soft_nll": 0.5436661450399293
  },
  "elapsed_seconds": 104.4
}


{
  "epoch": 2,
  "epochs": 4,
  "loss": 0.029320530981446307,
  "reward": 0.2746894745246714,
  "sigma": 0.30000000000000004,
  "validation": {
    "accuracy": 0.8784722222222222,
    "soft_nll": 0.5036433612306913
  },
  "elapsed_seconds": 208.3
}


{
  "epoch": 3,
  "epochs": 4,
  "loss": 0.010179993864666257,
  "reward": 0.2886141705740657,
  "sigma": 0.2,
  "validation": {
    "accuracy": 0.9184027777777778,
    "soft_nll": 0.47447674969832104
  },
  "elapsed_seconds": 312.2
}


{
  "epoch": 4,
  "epochs": 4,
  "loss": 0.08203417644189256,
  "reward": 0.29526991157197496,
  "sigma": 0.09999999999999998,
  "validation": {
    "accuracy": 0.9201388888888888,
    "soft_nll": 0.48136130968729657
  },
  "elapsed_seconds": 415.6
}


[train] saved checkpoint to artifacts/laya-model


## Independent calibration

Only `data/calibration.jsonl` is used to fit temperatures. The test split remains untouched.

In [ ]:
!python -m jevbro.calibrate \
  --model artifacts/laya-model \
  --data data/calibration.jsonl \
  --report artifacts/calibration-report.json

{
  "dataset": "data/calibration.jsonl",
  "question_types": {
    "choice": {
      "items": 144,
      "temperature": 1.498734942611036,
      "soft_nll_before": 0.7374953399657662,
      "soft_nll_after": 0.6804002531365282
    },
    "score": {
      "items": 144,
      "temperature": 1.6287801328949727,
      "soft_nll_before": 1.3781028410619767,
      "soft_nll_after": 1.3147292615857546
    },
    "noul": {
      "items": 288,
      "temperature": 1.7458086643728477,
      "soft_nll_before": 0.48028617071527613,
      "soft_nll_after": 0.4119666994814304
    }
  },
  "temperature": [
    1.498734942611036,
    1.6287801328949727,
    1.7458086643728477
  ]
}


## Final test

Run this after training and calibration. Do not tune the model against these results.

In [ ]:
!python -m jevbro.evaluate \
  --model artifacts/laya-model \
  --data data/test.jsonl \
  --device cuda \
  --report artifacts/test-report.json

[eval] 25/144 cases


[eval] 50/144 cases


[eval] 75/144 cases


[eval] 100/144 cases


[eval] 125/144 cases


{
  "model": "artifacts/laya-model",
  "dataset": "data/test.jsonl",
  "cases": 144,
  "decisions": 576,
  "accuracy": 0.7378472222222222,
  "ece": 0.05139812039260791,
  "score_mae": 0.5828680555555555,
  "by_primitive": {
    "choice": {
      "n": 144,
      "accuracy": 0.7638888888888888,
      "brier": 0.3026504162257423,
      "soft_accuracy": 0.608382543792877
    },
    "noul": {
      "n": 288,
      "accuracy": 0.8263888888888888,
      "brier": 0.21030252979166666,
      "soft_accuracy": 0.7369921736111114
    },
    "score": {
      "n": 144,
      "accuracy": 0.5347222222222222,
      "brier": 0.11889247880418871,
      "soft_accuracy": 0.3230003579961317
    }
  },
  "by_language": {
    "en": {
      "n": 384,
      "accuracy": 0.7057291666666666
    },
    "th": {
      "n": 192,
      "accuracy": 0.8020833333333334
    }
  }
}


In [ ]:
import laya, json
agent = laya.Agent('artifacts/laya-model', device='cuda')
from jevbro.questions import default_questions
samples = [
    'Agent wants to inspect git status without changing files',
    'เอเจนต์กำลังจะ deploy ระบบขึ้น production โดยยังไม่ได้รับอนุมัติ',
]
for text in samples:
    lang = 'th' if any('\u0e00' <= c <= '\u0e7f' for c in text) else 'en'
    result = agent.predict({'request': text}, default_questions(lang))
    print(text)
    print(json.dumps(result['answers'], ensure_ascii=False, indent=2))

Agent wants to inspect git status without changing files
{
  "action": {
    "type": "choice",
    "choice": "execute",
    "probabilities": {
      "execute": 0.7976,
      "ask_user": 0.1221,
      "reject": 0.0802
    },
    "confidence": 0.4179,
    "action": {
      "act_probability": 1.0
    }
  },
  "needs_review": {
    "type": "noul",
    "noul": 0.2155,
    "confidence": 0.7845,
    "action": {
      "act_probability": 1.0
    }
  },
  "prohibited": {
    "type": "noul",
    "noul": 0.12,
    "confidence": 0.88,
    "action": {
      "act_probability": 1.0
    }
  },
  "risk": {
    "type": "score",
    "score": 0.8395,
    "legend": {
      "0": "minimal: local/read-only/reversible with negligible external impact",
      "1": "low: limited scope and easy recovery",
      "2": "moderate: meaningful shared or external side effects",
      "3": "high: production, access, financial, publication, or difficult-to-reverse impact",
      "4": "critical: severe unauthorized, destruct

## Save artifacts to Drive

In [ ]:
DEST = Path('/content/drive/MyDrive/jev-my-bro-artifacts')
if DEST.exists():
    shutil.rmtree(DEST)
shutil.copytree(REPO / 'artifacts', DEST)
print('saved:', DEST)

saved: /content/drive/MyDrive/jev-my-bro-artifacts
